In [1]:
import numpy as np
from sklearn.linear_model import LinearRegression
import unittest
from matrix_helper import *

In [2]:
from ols_implementation import ols_fit

class TestOLSFitAgainstLibraries(unittest.TestCase):
    
    def setUp(self):
        # Create a synthetic dataset with some noise
        # X includes the intercept column of 1.0s
        self.X_list = [
            [1.0, 2.5, 1.2],
            [1.0, 1.2, 0.8],
            [1.0, 3.8, 2.5],
            [1.0, 4.1, 3.1],
            [1.0, 5.5, 4.0],
            [1.0, 0.5, 0.1]
        ]
        
        # y is a strict 2D column vector
        self.y_list = [
            [7.2],
            [4.5],
            [10.1],
            [11.0],
            [14.8],
            [2.1]
        ]
        
        self.X_arr = np.array(self.X_list)
        self.y_arr = np.array(self.y_list)

    def test_ols_against_sklearn(self):
        """
        Verify custom ols_fit beta coefficients and variance against scikit-learn.
        """
        # 1. Get custom results
        custom_beta, custom_sigma_sqr = ols_fit(self.X_list, self.y_list)
        custom_beta_flat = [b[0] for b in custom_beta] # Flatten to 1D for comparison
        
        # 2. Get sklearn results
        # fit_intercept=False because our X_list already has the leading column of 1s
        model = LinearRegression(fit_intercept=False)
        model.fit(self.X_arr, self.y_arr)
        
        # model.coef_ shape is (1, p) when y is (n, 1)
        sklearn_beta_flat = model.coef_[0] 
        
        # 3. Assert Beta coefficients match up to 5 decimal places
        np.testing.assert_almost_equal(custom_beta_flat, sklearn_beta_flat, decimal=5)
        
        # 4. Calculate sklearn's equivalent sigma squared for comparison
        sklearn_predictions = model.predict(self.X_arr)
        sklearn_rss = np.sum((self.y_arr - sklearn_predictions) ** 2)
        n = len(self.X_list)
        p_plus_1 = len(self.X_list[0])
        sklearn_sigma_sqr = sklearn_rss / (n - p_plus_1)
        
        # 5. Assert Sigma Squared matches
        self.assertAlmostEqual(custom_sigma_sqr, sklearn_sigma_sqr, places=5)

    def test_ols_against_numpy_lstsq(self):
        """
        Verify custom ols_fit beta coefficients against numpy's least squares solver.
        """
        custom_beta, _ = ols_fit(self.X_list, self.y_list)
        custom_beta_flat = [b[0] for b in custom_beta]
        
        # rcond=None suppresses a FutureWarning and uses the machine precision
        numpy_beta, residuals, rank, s = np.linalg.lstsq(self.X_arr, self.y_arr, rcond=None)
        numpy_beta_flat = numpy_beta.flatten()
        
        np.testing.assert_almost_equal(custom_beta_flat, numpy_beta_flat, decimal=5)

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

..
----------------------------------------------------------------------
Ran 2 tests in 0.003s

OK


In [3]:
from ols_implementation import hat_matrix

class TestHatMatrixAgainstLibraries(unittest.TestCase):
    
    def setUp(self):
        # Create a synthetic dataset
        # X includes the intercept column of 1.0s
        self.X_list = [
            [1.0, 2.5, 1.2],
            [1.0, 1.2, 0.8],
            [1.0, 3.8, 2.5],
            [1.0, 4.1, 3.1],
            [1.0, 5.5, 4.0],
            [1.0, 0.5, 0.1]
        ]
        
        self.X_arr = np.array(self.X_list)

    def test_hat_matrix_against_numpy(self):
        """
        Verify custom hat_matrix output against numpy matrix operations.
        """
        # 1. Get custom results
        custom_H = hat_matrix(self.X_list)
        
        # 2. Get numpy results
        # Formula: H = X * (X^T * X)^-1 * X^T
        X_t = self.X_arr.T
        X_t_X_inv = np.linalg.inv(X_t @ self.X_arr)
        numpy_H = self.X_arr @ X_t_X_inv @ X_t
        
        # 3. Assert matrix elements match up to 5 decimal places
        np.testing.assert_almost_equal(custom_H, numpy_H, decimal=5)

    def test_hat_matrix_properties(self):
        """
        Verify the mathematical properties of the hat matrix: symmetric and idempotent.
        """
        custom_H = hat_matrix(self.X_list)
        
        # Property 1: Symmetric (H = H^T)
        custom_H_t = mat_trans(custom_H)
        np.testing.assert_almost_equal(custom_H, custom_H_t, decimal=5)
        
        # Property 2: Idempotent (H * H = H)
        custom_H_squared = mat_mul(custom_H, custom_H)
        np.testing.assert_almost_equal(custom_H, custom_H_squared, decimal=5)

if __name__ == '__main__':
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

....
----------------------------------------------------------------------
Ran 4 tests in 0.004s

OK
